In [1]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [2]:
# connect to DB
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  dbname='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [19]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [3]:
# create m2_y table
try:
  cursor.execute(
    f'''
    create table m2_y as
    select
      no,
      replace(거래금액_만원,',','')::numeric * 10000 deal_amount,
      단지명 complex_nm,
      건축년도::numeric built_year,
      left(계약년월,4)::numeric - 건축년도::numeric building_age,
      층 floor,
      전용면적 deal_area,
      left(계약년월,4)::numeric contract_year,
      right(계약년월,2)::numeric contract_month,
      계약일::numeric contract_day,
      left(법정동코드,5) sig_cd,
      법정동코드 emd_cd,
      도로명 road_nm,
      concat(법정동코드,'1',본번,부번) pnu
    from 아파트_매매
    left join 법정동코드
    on 아파트_매매.시군구 = 법정동코드.법정동명
    where 해제사유발생일 = '-'
    '''
  )
except Exception as err:
  print(err)

오류:  "m2_y" 이름의 릴레이션(relation)이 이미 있습니다



In [3]:
# read m2_y
cursor.execute(
  'select * from m2_y'
)
y_df = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
x 구성

In [5]:
# 건축물 정보 bld_info
cursor.execute(
f'''
select
	pnu,
	sum(plat_area) plat_area,
	sum(arch_area) arch_area,
	sum(tot_area) tot_area,
	sum(elev_cnt) elev_cnt,
	sum(parklot_cnt) parklot_cnt
from building_info
group by 1
'''
)
bld_info = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [6]:
# 공시지가 lot_plp
cursor.execute(
  f'''
  select
    distinct on (pnu)
    base_year,
    pnu,
    amount plp_amt
  from public_land_price
  order by pnu,base_year desc
  '''
)
lot_plp = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)
del lot_plp['base_year']

In [7]:
# 지하철 접근성 - 최단거리 lot_subway_dist
cursor.execute(
  f'''
  select
    y.pnu,
    round(st_distance(
      y.geom_3857,
      subway_ent.geom_3857
    )) subway_dist
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (
    select st_collect(geom_3857) geom_3857
    from subway_ent
  ) subway_ent
  '''
)
lot_subway_dist = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [8]:
# 지하철 접근성 - 반경 1000m 내 역 개수 lot_subway_1000_cnt
cursor.execute(
  f'''
  select
    y.pnu,
    count(y.pnu) subway_1000_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (
    select
      station_nm,
      st_collect(geom_3857) geom_3857
    from subway_ent
    group by 1
  ) subway_ent
  where
    st_dwithin(
      y.geom_3857,
      subway_ent.geom_3857,
      1000
    )
  group by 1
  '''
)
lot_subway_1000_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [ ]:
# 유동인구 - 반경 500m 내 lot_walk_pop
cursor.execute(
  f'''
  select
    extract('year' from base_dt) base_year,
    y.pnu,
    sum(tot_cnt) walk_tot_cnt,
    sum(male_cnt) walk_male_cnt,
    sum(female_cnt) walk_female_cnt,
    sum(age_u20_cnt) walk_age_u20_cnt,
    sum(age_20_cnt) walk_age_20_cnt,
    sum(age_30_cnt) walk_age_30_cnt,
    sum(age_40_cnt) walk_age_40_cnt,
    sum(age_50_cnt) walk_age_50_cnt,
    sum(age_o50_cnt) walk_age_o50_cnt,
    sum(time_0810_cnt) walk_morning_cnt,
    sum(time_1113_cnt + time_1416_cnt) walk_afternoon_cnt,
    sum(time_1719_cnt + time_2022_cnt) walk_evening_cnt,
    sum(time_2307_cnt) walk_time_night_cnt
  from (
      select
        y.pnu,
        lot_polygon.geom_3857
      from (
        select
          distinct pnu
        from m2_y
      ) as y
      left join lot_polygon
      on y.pnu = lot_polygon.pnu
    ) y,
  walk_pop
  where
    walk_pop.base_dt < '2025-01-01' and
    st_dwithin(
      y.geom_3857,
      walk_pop.geom_3857,
      500
    )
  group by 1,2
  '''
)
lot_walk_pop = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [47]:
lot_walk_pop_2020 = lot_walk_pop[lot_walk_pop['base_year'] == 2021].copy()
lot_walk_pop_2020['base_year'] = 2020

In [48]:
lot_walk_pop_added = pd.concat(
  [lot_walk_pop,lot_walk_pop_2020]
)

In [18]:
# 거주인구 - 반경 500m 내
cursor.execute(
  f'''
  select
    pnu,
    sum(pop.pop_cnt) live_pop
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (
    select
      pop_cnt,
      geom_3857
    from live_pop
  ) as pop
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        500
      ),
      pop.geom_3857
    )
  group by 1
  '''
)
lot_live_pop = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [17]:
# 직장인구 - 반경 1000m 내
cursor.execute(
  f'''
  select
    pnu,
    sum(pop.pop_cnt) work_pop
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (
    select
      pop_cnt,
      geom_3857
    from work_pop
  ) as pop
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      pop.geom_3857
    )
  group by 1
  '''
)
lot_work_pop = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [9]:
# 인근 학교 개수 - 1. 어린이집 / 500m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_1_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '어린이집') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        500
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_1_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [11]:
# 인근 학교 개수 - 2. 유치원 / 500m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_2_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '유치원') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        500
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_2_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [12]:
# 인근 학교 개수 - 3. 초등학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_3_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '초등학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_3_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [13]:
# 인근 학교 개수 - 4. 중학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_4_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '중학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_4_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [14]:
# 인근 학교 개수 - 5. 고등학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_5_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '고등학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_5_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [15]:
# 인근 학교 개수 - 6. 대학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_6_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m2_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '대학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_6_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
m2_y + x 통합

In [46]:
y_df.contract_year.astype('string').value_counts()

contract_year
2020    80844
2024    54884
2021    41946
2023    34124
2022    12045
Name: count, dtype: Int64

In [ ]:
lot_walk_pop

,base_year,pnu,walk_tot_cnt,walk_male_cnt,walk_female_cnt,walk_age_u20_cnt,walk_age_20_cnt,walk_age_30_cnt,walk_age_40_cnt,walk_age_50_cnt,walk_age_o50_cnt,walk_morning_cnt,walk_afternoon_cnt,walk_evening_cnt,walk_time_night_cnt
0,2021,1111010100100560045,5297255.634234879376542,2445559.76,2076553.57,1304909.44,532254.69,581656.52,696157.70,614095.22,79304029.02,787214.33,1586556.33,951075.34,1197267.58
1,2022,1111010100100560045,7083005.802004519148,3677034.14,3405971.18,2187693.24,695353.24,811575.53,989992.93,970986.66,142740391.13,1225761.85,2725591.97,1392901.82,1738750.57
2,2023,1111010100100560045,6980024.63256664629975,3696563.72,3283460.85,2391314.12,619754.43,741765.24,890412.94,910900.24,142587774.64,1228741.11,2652623.49,1337337.47,1761321.72
3,2024,1111010100100560045,4534740.91,2296941.52,2237799.38,1492261.79,441664.23,432166.62,594266.33,593760.96,97793983.94,826869.69,1754442.12,884726.38,1067899.57
4,2025,1111010100100560045,544868.21,240068.00,304800.21,91624.81,57341.61,76922.65,77465.09,83299.88,15696728.11,84960.74,192071.46,116180.97,151370.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40290,2021,1174011000107170000,25779929.885896625052,9587279.70,10889663.48,5093060.37,2375077.61,2593260.41,3459276.95,2987114.13,396915364.60,2637336.57,5425998.86,5766131.85,6647475.41
40291,2022,1174011000107170000,24969023.505138434011,11916115.66,13052908.00,5952754.27,2694798.60,3071727.37,4135293.09,3823962.38,529025957.59,3224556.40,6557999.42,6954812.18,8231656.21
40292,2023,1174011000107170000,25704749.18821921960686,12080566.96,13624182.26,5941436.02,2770044.79,3151915.62,4487761.21,3766094.11,558731090.97,3224027.00,6581136.61,7051152.55,8848433.34
40293,2024,1174011000107170000,23960554.92,11184135.83,12776419.10,5513311.13,2569841.58,2838071.95,4115471.37,3603160.56,532064755.68,2907748.65,6250356.19,6434843.98,8367607.68


In [49]:
m2_total_df = y_df.merge(
  bld_info,
  how='left',
  on='pnu'
).merge(
  lot_plp,
  how='left',
  on='pnu'
).merge(
  lot_subway_dist,
  how='left',
  on='pnu'
).merge(
  lot_subway_1000_cnt,
  how='left',
  on='pnu'
).merge(
  lot_walk_pop_added,
  how='left',
  left_on=['contract_year','pnu'],
  right_on=['base_year','pnu'],
).merge(
  lot_live_pop,
  how='left',
  on='pnu'
).merge(
  lot_work_pop,
  how='left',
  on='pnu'
).merge(
  school_type_1_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_2_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_3_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_4_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_5_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_6_cnt,
  how='left',
  on='pnu'
)

In [52]:
m2_total_df.to_sql(
  'm2_total',
  engine,
  if_exists='replace',
  index=False
)

200

---
전달용 데이터 저장하기

In [53]:
m2_total_df.to_csv(
  'm2_data_06.22.csv',
  sep=',',
  index=False
)